# 5. Resultados da programação não linear

Leitura da bateria: o que cada método consegue na PNL cinemática, e como isso
se compara ao paper (\(t_f\approx 1{,}180\) s, obstáculo com \(N=3\)).


In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
for cand in [here, here.parent, here / "T2", here.parent / "T2"]:
    if (cand / "src" / "parametros.py").exists():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        break
from src.ambiente import preparar
T2 = preparar()
print("T2 =", T2)


In [ ]:
import pickle
from src.paths import saida_dir

with open(saida_dir() / "bateria.pkl", "rb") as f:
    payload = pickle.load(f)
resumo = payload["resumo"]
melhor = payload["melhor"]

print(f"{'método':20s} {'ok':>6s} {'tf min':>8s} {'tf méd':>8s} {'tf máx':>8s} {'CPU méd':>8s}")
for m, s in resumo.items():
    print(f"{m:20s} {s['n_sucesso']}/{s['n']:<3d} {s['tf_min']:8.4f} {s['tf_media']:8.4f} "
          f"{s['tf_max']:8.4f} {s['cpu_media']:8.3f}")
print(f"\nmelhor: {melhor['metodo']} / {melhor['chute']}  tf={melhor['tf']:.4f} s")
print("paper, cinemática com obstáculo N=3: tf ≈ 1.180 s")


## Leitura

- **Pontos interiores** (`trust-constr`) são o único método que devolveu
  soluções viáveis com \(t_f\) competitivo. A Jacobiana analítica das restrições
  entra direto no solver. Partidas diferentes caem em mínimos locais distintos
  (o C-space não é convexo): a média de \(t_f\) fica acima do mínimo.
- **Pattern search** (Hooke–Jeeves) trabalha na penalidade quadrática, sem
  derivada, em \(\mathbb{R}^{21}\). Com o orçamento de iterações da bateria não
  restaurou viabilidade a partir de chutes que colidem, e a partir do desvio
  viável não reduziu \(t_f\) o bastante sem violar o disco.
- **Newton** na mesma penalidade, mesmo com Hessiana analítica, sofre com a
  degenerescência de \(\nabla^2\Phi\) quando as igualdades estão quase satisfeitas
  (posto baixo) e com a não suavidade de \(\max(g,0)\). Os cinco chutes
  terminaram inviáveis.
- **Paper.** \(t_f\approx 1{,}18\) s com colocação LGR e malha bem mais fina.
  Aqui \(K=10\) e aceleração constante por peça; o melhor \(t_f\) viável deve ser
  lido na mesma ordem de grandeza, não como reprodução do número do artigo.
  A passagem é pela esquerda do disco no C-space, como no caso \(N=3\).
